# Visualizações Interativas - Demonstrações Financeiras CVM 2024

Neste notebook, todos os gráficos são interativos via Plotly.  
Você poderá explorar hover, filtros e zoom nos indicadores financeiros

In [1]:
!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.0 MB/s eta 0:00:00


In [2]:
# 0️. Importação de bibliotecas e conexão
import pandas as pd
import numpy as np
import plotly.express as px
from sqlalchemy import create_engine, text

engine_cvm = create_engine('sqlite:///data_cvm.db')

def query_sql(query_string):
    with engine_cvm.connect() as con:
        return pd.read_sql(text(query_string), con=con)

## 1️. Top 10 Empresas por ROE

Exibe as empresas que entregaram maior retorno sobre patrimônio líquido em 2024.

In [3]:
# Top 10 Empresas por ROE – gráfico limpo, sem ruídos, sempre 10 barras
import plotly.express as px

# 1️. Consulta básica
query_top_roe = """
SELECT DENOM_CIA, ROE, PORTE_EMPRESA
FROM empresas_cvm
WHERE ROE IS NOT NULL
"""
df_top_roe = query_sql(query_top_roe)

# 2️. Filtrar ruídos extremos
df_top_roe = df_top_roe[(df_top_roe['ROE'] > -50) & (df_top_roe['ROE'] < 200)]

# 3️. Ordenar e pegar top 10
df_top_roe = df_top_roe.sort_values('ROE', ascending=False).head(10)

# 4️. Plot interativo
fig = px.bar(
    df_top_roe,
    x='ROE',
    y='DENOM_CIA',
    color='PORTE_EMPRESA',
    orientation='h',
    text='ROE',
    color_discrete_sequence=px.colors.sequential.Viridis
)

fig.update_layout(
    title='Top 10 Empresas por ROE (Rentabilidade sobre Patrimônio)',
    xaxis_title='ROE (%)',
    yaxis_title='Empresa',
    yaxis=dict(autorange="reversed"),
    legend_title='Porte da Empresa'
)

fig.show()

**Insight:**  
Permite identificar rapidamente as empresas mais eficientes para o acionista, visualizando porte e ROE em hover.

## 2️. Receita de Venda x Lucro Líquido (Interativo)
Mostra correlação entre receita e lucro, permitindo identificar empresas com alta receita mas baixa rentabilidade.

In [4]:
query_relacao = """
SELECT RECEITA_VENDA, LUCRO_LÍQUIDO, DENOM_CIA, PORTE_EMPRESA, ATIVO_TOTAL
FROM empresas_cvm
"""
df_rel = query_sql(query_relacao)

fig = px.scatter(
    df_rel,
    x='RECEITA_VENDA',
    y='LUCRO_LÍQUIDO',
    color='PORTE_EMPRESA',
    size='ATIVO_TOTAL',
    hover_name='DENOM_CIA',
    log_x=True,
    size_max=50,
    title='Receita de Venda x Lucro Líquido (Escala Log)'
)
fig.update_layout(
    xaxis_title='Receita de Venda (R$) - log',
    yaxis_title='Lucro Líquido (R$)',
)
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.show()

**Insight:**  
Ajuda a identificar empresas grandes com baixa margem ou prejuízo, destacando risco operacional.

## 3️. Distribuição da Margem Líquida (Interativo)

Visualiza como as margens líquidas se distribuem entre empresas, por porte.

In [5]:
query_margem = """
SELECT MARGEM_LÍQUIDA, PORTE_EMPRESA
FROM empresas_cvm
WHERE MARGEM_LÍQUIDA BETWEEN -50 AND 100
"""
df_margem = query_sql(query_margem)

fig = px.histogram(
    df_margem,
    x='MARGEM_LÍQUIDA',
    color='PORTE_EMPRESA',
    nbins=25,
    barmode='overlay',
    marginal='box',
    title='Distribuição da Margem Líquida das Empresas (Por Porte)',
    labels={'MARGEM_LÍQUIDA':'Margem Líquida (%)'}
)
fig.add_vline(x=df_margem['MARGEM_LÍQUIDA'].mean(), line_dash="dash", line_color="black", annotation_text="Média Geral")
fig.show()

**Insight:**  
Permite ver concentração de margens positivas e negativas, identificar porte e avaliar eficiência operacional.

## 4️. Comparação de ROE por Porte de Empresa (Interativo)

Boxplot interativo com swarmplot embutido para visualizar dispersão e outliers.

In [6]:
query_geral = """
SELECT PORTE_EMPRESA, ROE, DENOM_CIA
FROM empresas_cvm
"""
df_para_grafico = query_sql(query_geral)

fig = px.box(
    df_para_grafico,
    x='PORTE_EMPRESA',
    y='ROE',
    points="all",
    color='PORTE_EMPRESA',
    title='Comparação de ROE por Porte de Empresa',
    labels={'ROE':'ROE (%)', 'PORTE_EMPRESA':'Porte da Empresa'}
)
fig.update_yaxes(range=[-20, 60])
fig.show()

**Insight:**  
Permite comparar dispersão, outliers e eficiência entre diferentes portes de empresa, destacando risco e oportunidades para acionistas.

# Dashboard Financeiro - Empresas CVM 2024

Este dashboard apresenta uma visão consolidada das empresas listadas na CVM em 2024, com base nos indicadores financeiros calculados:

- **ROE** (Retorno sobre Patrimônio)
- **ROA** (Retorno sobre Ativos)
- **Margem Líquida**
- **Classificação de porte das empresas**

O objetivo é fornecer insights sobre rentabilidade, eficiência operacional e risco financeiro.

In [7]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# --- Preparando os dados ---
# Top 10 ROE
query_top_roe = "SELECT DENOM_CIA, ROE FROM empresas_cvm WHERE ROE IS NOT NULL ORDER BY ROE DESC LIMIT 10"
df_top_roe = query_sql(query_top_roe)
df_top_roe = df_top_roe[(df_top_roe['ROE'] > -50) & (df_top_roe['ROE'] < 200)]

# Receita x Lucro
query_rel = "SELECT RECEITA_VENDA, LUCRO_LÍQUIDO, DENOM_CIA FROM empresas_cvm"
df_rel = query_sql(query_rel)

# Margem Líquida
query_margem = "SELECT MARGEM_LÍQUIDA FROM empresas_cvm WHERE MARGEM_LÍQUIDA BETWEEN -50 AND 100"
df_margem = query_sql(query_margem)

# ROE por Porte
query_porte = "SELECT PORTE_EMPRESA, ROE FROM empresas_cvm WHERE ROE IS NOT NULL"
df_porte = query_sql(query_porte)
df_porte = df_porte[(df_porte['ROE'] > -50) & (df_porte['ROE'] < 200)]

# --- Criando subplots ---
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Top 10 ROE", "Receita x Lucro", "Distribuição Margem Líquida", "ROE por Porte"),
    horizontal_spacing=0.15,
    vertical_spacing=0.15
)

# Gráfico 1: Top 10 ROE
fig.add_trace(
    go.Bar(
        x=df_top_roe['ROE'],
        y=df_top_roe['DENOM_CIA'],
        orientation='h',
        marker_color='teal',
        name='Top ROE'
    ),
    row=1, col=1
)

# Gráfico 2: Receita x Lucro
fig.add_trace(
    go.Scatter(
        x=df_rel['RECEITA_VENDA'],
        y=df_rel['LUCRO_LÍQUIDO'],
        mode='markers',
        marker=dict(size=8, color='orange', opacity=0.6),
        name='Receita x Lucro'
    ),
    row=1, col=2
)

# Gráfico 3: Margem Líquida
fig.add_trace(
    go.Histogram(
        x=df_margem['MARGEM_LÍQUIDA'],
        nbinsx=20,
        marker_color='purple',
        name='Margem Líquida'
    ),
    row=2, col=1
)

# Gráfico 4: ROE por Porte
for porte in df_porte['PORTE_EMPRESA'].unique():
    fig.add_trace(
        go.Box(
            y=df_porte[df_porte['PORTE_EMPRESA']==porte]['ROE'],
            name=porte
        ),
        row=2, col=2
    )

# --- Layout final ---
fig.update_layout(
    height=800, width=1000,
    title_text="Dashboard Financeiro – Empresas CVM 2024",
    showlegend=False
)

fig.show()

## Observações do Dashboard

- As **maiores empresas em ROE** não são necessariamente as de maior porte.  
- Algumas empresas com **alta receita apresentam margem negativa**, indicando possível ineficiência operacional.  
- O **ROE por porte** mostra que empresas médias podem ter operações mais enxutas e rentáveis que grandes empresas.  
- Empresas com **patrimônio líquido negativo** apresentam maior risco financeiro para investidores.